In [ ]:
import os
import json
import torch
from unsloth import FastVisionModel, is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
from transformers import Trainer, TrainingArguments
from torch.utils.data import Dataset as TorchDataset
from dotenv import load_dotenv

load_dotenv()

os.environ["WANDB_DISABLED"] = "true"

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
TRAIN_DATASET = os.getenv("train_json")
OUTPUT_DIR = os.getenv("output")

In [ ]:
class ConversationDataset(TorchDataset):
    def __init__(self, samples):
        self.samples = samples
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        return self.samples[idx]

print("[INFO] Memuat & memproses dataset JSONL...")
fixed_samples = []
skipped = 0

with open(TRAIN_DATASET, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        
        sample = json.loads(line)
        new_messages = []
        valid = True
        
        for message in sample["messages"]:
            role = message["role"]
            content = message["content"]
            
            if role == "assistant":
                text = str(content)
                new_messages.append({"role": "assistant", "content": text})
            
            elif role == "user":
                new_content = []
                for item in content:
                    if isinstance(item, dict) and item.get("type") == "image":
                        img_path = item["image"]
                        # Cek apakah gambar 
                        if not os.path.exists(img_path):
                            print(f"[WARNING] Gambar hilang: {img_path}")
                            valid = False
                            break
                        new_content.append({"type": "image", "image": img_path})
                    else:
                        new_content.append(item)
                
                if not valid:
                    break
                new_messages.append({"role": "user", "content": new_content})
        
        if valid:
            fixed_samples.append({"messages": new_messages})
        else:
            skipped += 1

print(f"[INFO] Total data VALID yang siap di-training: {len(fixed_samples)}")
print(f"[INFO] Data dilewati (gambar hilang): {skipped}")

# Split Dataset (90% Train, 10% Eval)
import random
random.seed(3407)
random.shuffle(fixed_samples)

split_idx = int(len(fixed_samples) * 0.9)
train_dataset = ConversationDataset(fixed_samples[:split_idx])
eval_dataset  = ConversationDataset(fixed_samples[split_idx:])

print(f"[INFO] Train split: {len(train_dataset)} | Eval split: {len(eval_dataset)}")

In [ ]:
print("[INFO] Memuat Model Qwen3-VL-4B...")
model, tokenizer = FastVisionModel.from_pretrained(
    "Qwen/Qwen3-VL-4B-Instruct",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

print("[INFO] Memasang LoRA Adapter...")
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True, 
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=3407,
)

In [ ]:
trainer = Trainer(
    model=model,
    data_collator=UnslothVisionDataCollator(model, tokenizer,),
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=2, 
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=20, 
        learning_rate=2e-4,
        fp16=not is_bf16_supported(),
        bf16=is_bf16_supported(),
        logging_steps=5,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=50,
        save_total_limit=3,
        load_best_model_at_end=True, 
        metric_for_best_model="eval_loss",
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        report_to="none",
        remove_unused_columns=False,
    ),
)

print("[INFO] Memulai Fine-Tuning...")
trainer_stats = trainer.train()

print(f"\n[INFO] Training selesai dalam waktu: {trainer_stats.metrics['train_runtime']:.2f} detik")

print(f"\n[INFO] Menyimpan LoRA Adapter terbaik ke: {OUTPUT_DIR}")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("\n[SELESAI] Model siap digunakan!")